In [ ]:
# Ensure Hugging Face and Torch caches are set before imports
import os
cwd = os.getcwd()

cache_dir = "/home/sazhang/.cache"
hf_cache = f"{cache_dir}/huggingface"
torch_cache = f"{cache_dir}/torch"

os.environ["HF_HOME"] = hf_cache
os.environ["HF_HUB_CACHE"] = f"{hf_cache}/hub"
os.environ["TRANSFORMERS_CACHE"] = f"{hf_cache}/transformers"
os.environ["TORCH_HOME"] = torch_cache

# If huggingface_hub was imported earlier in this kernel, reload constants
import importlib
import huggingface_hub.constants as hf_const
importlib.reload(hf_const)
print("HF_HOME:", hf_const.HF_HOME)
print("HF_HUB_CACHE:", hf_const.HF_HUB_CACHE)
print("TORCH_HOME:", os.environ["TORCH_HOME"])

# os.environ["MPLCONFIGDIR"] = "/home/sazhang/.config/matplotlib"


HF_HOME: /home/sazhang/.cache/huggingface
HF_HUB_CACHE: /home/sazhang/.cache/huggingface/hub
TORCH_HOME: /home/sazhang/.cache/torch


In [2]:
import sys
sys.path.append(f'{cwd}/satclip/satclip') # add satclip module from satclip repo to path

import torch
from load import get_satclip

mkdir -p failed for path /afs/csail.mit.edu/u/s/sazhang/.config/matplotlib: [Errno 13] Permission denied: '/afs/csail.mit.edu/u/s/sazhang/.config/matplotlib'
Matplotlib created a temporary cache directory at /tmp/matplotlib-l57rdnen because there was an issue with the default path (/afs/csail.mit.edu/u/s/sazhang/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.
/home/sazhang/miniconda3/envs/downscaling/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Pretrained SatCLIP models are hosted on Hugging Face, where we can easily access them. Here, for example, we use the $L=40$ version with a ResNet18 backbone.

In [3]:
from huggingface_hub import hf_hub_download
from load import get_satclip
import torch


device = "cuda" if torch.cuda.is_available() else "cpu"

c = torch.randn(32, 2)  # Represents a batch of 32 locations (lon/lat)

model = get_satclip(
    hf_hub_download("microsoft/SatCLIP-ResNet18-L40", "satclip-resnet18-l40.ckpt", cache_dir=cache_dir),
    device=device,
)  # Only loads location encoder by default
model.eval()
with torch.no_grad():
    emb = model(c.double().to(device)).detach().cpu()

using pretrained moco resnet18
Downloading: "https://hf.co/torchgeo/resnet18_sentinel2_all_moco/resolve/5b8cddc9a14f3844350b7f40b85bcd32aed75918/resnet18_sentinel2_all_moco-59bfdff9.pth" to /home/sazhang/.cache/torch/hub/checkpoints/resnet18_sentinel2_all_moco-59bfdff9.pth


100%|██████████| 42.8M/42.8M [00:00<00:00, 348MB/s]


In [7]:
emb.shape

torch.Size([32, 256])